# P4 - Clinical Persona + Structured Causal CoT

**Prompt ID:** `P4_clinical_persona_cot`  
**LLM:** `gpt-4.1-mini` (OpenAI)  
**Embedder:** `text-embedding-3-small`

Adapted from MedCoT-RAG (Wang et al., 2024). Adds a clinical evidence reviewer persona plus a 4-stage workflow: Identify Findings -> Assess Quality -> Weigh Evidence -> Decide. Hypothesis: gives the best faithfulness, esp. when combined with Context Reranking.

## Pilot vs Full mode

By default this notebook runs in **pilot mode** to keep API cost low:
- `MAX_SAMPLES = 100` (vs 500 in main experiments)
- `METHODS = ["baseline"]` (vs all 4 in main experiments)

After reviewing pilot results across all 5 prompts in the analysis notebook (`06_analysis.ipynb`), increase to full run by changing the two variables in the next cell.


In [1]:
import os
import shared

# Set env var
# os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"

# Update shared module supaya pakai key baru
shared.OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

# Verifikasi
print(f'Key set: {shared.OPENAI_API_KEY[:10]}...{shared.OPENAI_API_KEY[-4:]}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Key set: sk-proj-q6...FXQA


In [2]:
# ============================================================
# EXPERIMENT CONFIG - adjust here
# ============================================================
PROMPT_ID    = 'P4_clinical_persona_cot'

# Pilot mode (cheap, fast)
MAX_SAMPLES  = 100
METHODS      = ['baseline']

# Full run (uncomment when ready):
# MAX_SAMPLES = 500
# METHODS     = ['baseline', 'qr', 'cr', 'qr_cr']


In [3]:
# ============================================================
# Setup - import shared utilities + the chosen prompt
# ============================================================
import sys
from pathlib import Path

# Make sure the prompt_experiment folder is on sys.path
sys.path.insert(0, str(Path('.').resolve()))

import shared
from prompts import PROMPTS, PROMPT_MAX_TOKENS

PROMPT_TEMPLATE = PROMPTS[PROMPT_ID]
MAX_TOKENS      = PROMPT_MAX_TOKENS[PROMPT_ID]

print(f'Prompt    : {PROMPT_ID}')
print(f'Length    : {len(PROMPT_TEMPLATE)} chars')
print(f'Max tokens: {MAX_TOKENS}')
print(f'Samples   : {MAX_SAMPLES}')
print(f'Methods   : {METHODS}')


Prompt    : P4_clinical_persona_cot
Length    : 1476 chars
Max tokens: 500
Samples   : 100
Methods   : ['baseline']


In [4]:
# Load PubMedQA, BM25 index, Chroma collection, CrossEncoder
# (uses indexes already built by the main notebooks)
shared.load_everything(max_samples=MAX_SAMPLES)


Loaded PubMedQA: 100 samples
Loaded BM25 index: 1706 document chunks
Loaded Chroma collection: 1706 vectors


Loading weights: 100%|███████████████████████| 105/105 [00:00<00:00, 338.59it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2


(Dataset({
     features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
     num_rows: 100
 }),
 [Document(text='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', pubid='21645374', question='Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', section_label='BACKGROUND', answer='Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during de

## Smoke test


In [5]:
# Smoke test on 1 sample to make sure the pipeline works end-to-end
_data, *_ = shared.load_everything(max_samples=MAX_SAMPLES)
_s   = _data[0]
_q   = _s['question']
_gt  = _s['final_decision']
_ret, _used_q = shared.run_baseline(_q)
_ans = shared.generate_answer(_q, _ret, PROMPT_TEMPLATE, max_tokens=MAX_TOKENS)
_pred = shared.extract_label(_ans)
print(f'Q: {_q}')
print(f'GT: {_gt} | Pred: {_pred} | Match: {_gt == _pred}')
print(f'Answer preview:')
print(_ans[:500])


Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
GT: yes | Pred: no | Match: False
Answer preview:
(A) IDENTIFY FINDINGS  
- Abstracts [1] and [2] are relevant to the research question about mitochondria's role in lace plant leaf remodeling during programmed cell death (PCD).  
- [1] Background: Lace plant leaves develop perforations via PCD, which occurs in a spatial pattern within areoles. The role of mitochondria in animal PCD is known but less studied in plants.  
- [2] Results: In vivo study of mitochondrial dynamics during PCD in lace plant leaves. Cells categorized by PCD stage (NPCD, 


## Phase 1 - generate answers for each method


In [6]:
phase1_paths = {}
for method in METHODS:
    print(f'\n=== Phase 1: prompt={PROMPT_ID}, method={method} ===')
    p1 = shared.run_phase1(
        prompt_id=PROMPT_ID,
        prompt_template=PROMPT_TEMPLATE,
        max_tokens=MAX_TOKENS,
        method=method,
        max_samples=MAX_SAMPLES,
    )
    phase1_paths[method] = p1



=== Phase 1: prompt=P4_clinical_persona_cot, method=baseline ===
  Start  P4_clinical_persona_cot__baseline_openai: 0/100
    [ 10/100] acc=20.0% | ETA 12.1m
    [ 20/100] acc=35.0% | ETA 11.5m
    [ 30/100] acc=36.7% | ETA 9.5m
    [ 40/100] acc=42.5% | ETA 7.8m
    [ 50/100] acc=42.0% | ETA 6.7m
    [ 60/100] acc=41.7% | ETA 5.3m
    [ 70/100] acc=44.3% | ETA 3.9m
    [ 80/100] acc=45.0% | ETA 2.5m
    [ 90/100] acc=44.4% | ETA 1.3m
    [100/100] acc=45.0% | ETA 0.0m


## Phase 2 - evaluate (4 metrics)


In [7]:
phase2_paths = {}
for method, p1_path in phase1_paths.items():
    print(f'\n=== Phase 2: prompt={PROMPT_ID}, method={method} ===')
    p2 = shared.run_phase2(p1_path, max_samples=MAX_SAMPLES)
    phase2_paths[method] = p2



=== Phase 2: prompt=P4_clinical_persona_cot, method=baseline ===
  Phase2 P4_clinical_persona_cot__baseline_openai_phase1: 0/100 done, 100 remaining.
    [ 10/100] f=0.739 cr=0.675 ar=0.806 cp=0.800 | ETA 61.4m
    [ 20/100] f=0.791 cr=0.754 ar=0.803 cp=0.759 | ETA 50.4m
    [ 30/100] f=0.814 cr=0.803 ar=0.808 cp=0.750 | ETA 41.7m
    [ 40/100] f=0.827 cr=0.827 ar=0.804 cp=0.750 | ETA 35.3m
    [ 50/100] f=0.822 cr=0.832 ar=0.808 cp=0.749 | ETA 30.0m
    [ 60/100] f=0.829 cr=0.824 ar=0.801 cp=0.754 | ETA 24.0m
    [ 70/100] f=0.828 cr=0.820 ar=0.795 cp=0.748 | ETA 17.7m
    [ 80/100] f=0.826 cr=0.812 ar=0.797 cp=0.731 | ETA 11.9m
    [ 90/100] f=0.817 cr=0.799 ar=0.796 cp=0.713 | ETA 6.0m
    [100/100] f=0.815 cr=0.799 ar=0.798 cp=0.725 | ETA 0.0m


## Summary


In [8]:
print('=' * 80)
print(f'PROMPT: {PROMPT_ID}    (n={MAX_SAMPLES} per method)')
print('=' * 80)
for method, p2_path in phase2_paths.items():
    shared.print_summary(p2_path)


PROMPT: P4_clinical_persona_cot    (n=100 per method)
  P4_clinical_persona_cot__baseline_openai_phase2 | n=100 | acc=0.450 f=0.815 cr=0.799 ar=0.798 cp=0.725
        yes: 25/58 = 43.1%
         no: 12/26 = 46.2%
      maybe: 8/16 = 50.0%
